# LightGBM as a challenger

Curious whether LightGBM beats XGB on the home credit window. Parked.
Initial run showed roughly the same AUC; native categorical handling
would need wiring through `features.py` to skip the one-hot for those
columns, and I haven't finished that.

In [ ]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

from modelgate.data import load_raw, split_into_weeks, prepare_window_for_training
from modelgate.features import engineer, split_x_y, build_preprocessor

In [ ]:
df = load_raw()
weeks = split_into_weeks(df)
train, val = prepare_window_for_training(weeks, end_week=3)
x_train, y_train = split_x_y(engineer(train))
x_val, y_val = split_x_y(engineer(val))
pre = build_preprocessor(x_train)
x_train_t = pre.fit_transform(x_train)
x_val_t = pre.transform(x_val)

In [ ]:
# TODO: pass categorical_feature= and skip the one-hot encoding above
#       for those columns. native LGBM categoricals are nicer than
#       a 200-column one-hot. not done.
clf = lgb.LGBMClassifier(
    n_estimators=400, learning_rate=0.05, num_leaves=31,
    objective='binary', random_state=42, verbose=-1,
)
clf.fit(x_train_t, y_train, eval_set=[(x_val_t, y_val)])
p = clf.predict_proba(x_val_t)[:, 1]
print('val AUC:', roc_auc_score(y_val, p))

Initial result without tuning was 0.752, basically tied with XGB on the
same window. Probably worth tuning before declaring a winner. Coming
back to this later.